In [33]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
import pandas as pd
import numpy as np

from relbench.base import BaseTask, Dataset
from relbench.datasets import get_dataset
from relbench.tasks import get_task, get_task_names

In [35]:
from rtgl.base import Table
from rtgl.converter import SConverter, TConverter

## Helper Functions

In [36]:
def get_timestamps(dataset: Dataset, 
                   timedelta: pd.Timedelta, 
                   num_eval_timestamps: int, 
                   split: str) -> "pd.Series[pd.Timestamp]":
    db = dataset.get_db(upto_test_timestamp=(split != "test"))

    if split == "train":
        start = dataset.val_timestamp - timedelta
        end = db.min_timestamp
        freq = -timedelta
    elif split == "val":
        start = dataset.val_timestamp
        end = min(
            dataset.val_timestamp
            + timedelta * (num_eval_timestamps - 1),
            dataset.test_timestamp - timedelta,
            )
        freq = timedelta
    elif split == "test":
        start = dataset.test_timestamp
        end = min(
            dataset.test_timestamp
            + timedelta * (num_eval_timestamps - 1),
            db.max_timestamp - timedelta,
            )
        freq = timedelta
    else:
        pass

    timestamps = pd.date_range(start=start, end=end, freq=freq)
    return timestamps

In [37]:
def merge_dataframes(df_orig: pd.DataFrame,
                     df_inj: pd.DataFrame, 
                     type: str) -> None:
    # normalization if LIST_DISTINCT was used in the query
    def normalize(x):
        if isinstance(x, (list, np.ndarray, tuple)):
            return tuple(sorted(x))
        return x

    df_orig['label'] = df_orig['label'].apply(normalize)
    df_inj['label'] = df_inj['label'].apply(normalize)

    merged = pd.merge(
        df_orig,
        df_inj,
        on=['fk', 'timestamp', 'label'] if type == "tmp" else ['fk', 'label'],
        how='outer',
        suffixes=('_orig', '_inj'),
        indicator=True
    )

    print(f"Query without INJECTION:\n {merged[merged['_merge'] == 'left_only']}")
    print(f"Query with INJECTION:\n {merged[merged['_merge'] == 'right_only']}")
    print(f"In both:\n {merged[merged['_merge'] == 'both']}")

In [38]:
def check_correctness(dataset: Dataset,
                      task: BaseTask,
                      split: str,
                      query_orig: str,
                      query_inj: str,
                      type: str) -> None:
    if type == "tmp":
        timestamps = get_timestamps(dataset, task.timedelta, task.num_eval_timestamps, split)
        converter = TConverter(dataset.get_db(upto_test_timestamp=(split != "test")), timestamps) 
    else:
        converter = SConverter(dataset.get_db(upto_test_timestamp=(split != "test")))
    
    table_orig = converter.convert(query_orig, execute=True)
    table_inj = converter.convert(query_inj, execute=True)

    df_orig = table_orig.df
    df_inj = table_inj.df

    print(f"------------------- START {split.upper()} -------------------")
    print(f"FKeys without INJECTION: {table_orig.fkey_col_to_pkey_table}")
    print(f"PKey without INJECTION: {table_orig.pkey_col}")
    print(f"TimeCol without INJECTION: {table_orig.time_col}")
    print(f"FKeys with INJECTION: {table_inj.fkey_col_to_pkey_table}")
    print(f"PKey with INJECTION: {table_inj.pkey_col}")
    print(f"TimeCol with INJECTION: {table_inj.time_col}")
    merge_dataframes(df_orig, df_inj, type)
    print(f"------------------- END {split.upper()} ---------------------")

## Stack-Exchange Q&A Website Dataset

In [39]:
dataset_stack = get_dataset(name="rel-stack", download=False)
db_stack = dataset_stack.get_db()

In [8]:
db_stack.table_dict['votes']

Table(df=
              Id  UserId  PostId  VoteTypeId CreationDate
0              0    <NA>       2           2   2009-02-02
1              1    <NA>       6           2   2009-02-02
2              2    <NA>       6           2   2009-02-02
3              3    <NA>       6           2   2009-02-02
4              4    <NA>       6           2   2009-02-02
...          ...     ...     ...         ...          ...
1317871  1317871    <NA>    <NA>          15   2021-01-01
1317872  1317872    <NA>   48550           2   2021-01-01
1317873  1317873    <NA>  217931           2   2021-01-01
1317874  1317874    <NA>  255319           1   2021-01-01
1317875  1317875    <NA>  232649           2   2021-01-01

[1317876 rows x 5 columns],
  fkey_col_to_pkey_table={'PostId': 'posts', 'UserId': 'users'},
  pkey_col=Id,
  time_col=CreationDate)

In [9]:
db_stack.table_dict['posts']

Table(df=
            Id  OwnerUserId  PostTypeId  AcceptedAnswerId  ParentId  \
0            0        76449           1                 1      <NA>   
1            1          957           2              <NA>         0   
2            2         <NA>           2              <NA>         0   
3            3          884           2              <NA>         0   
4            4         <NA>           2              <NA>         0   
...        ...          ...         ...               ...       ...   
333888  333888       134654           2              <NA>    333796   
333889  333889       216615           2              <NA>    332888   
333890  333890       216615           2              <NA>    320669   
333891  333891       211410           1              <NA>      <NA>   
333892  333892       255358           2              <NA>     27860   

       OwnerDisplayName                                              Title  \
0                Tawani  What do you call an average that d

In [41]:
task_stack_post_votes = get_task("rel-stack", "post-votes", download=False)

In [42]:
query_orig = """
    PREDICT COUNT_DISTINCT(votes.* WHERE votes.votetypeid == 2, 0, 91, DAYS)
    FOR EACH posts.* WHERE posts.PostTypeId == 1
                        AND posts.OwnerUserId IS NOT NULL
                        AND posts.OwnerUserId != -1
    ;
"""

In [43]:
query_inj = """
    PREDICT COUNT_DISTINCT(
    [
     SELECT 
        v.*
     FROM 
        votes v
     WHERE 
        v.votetypeid = ANY(ARRAY[2])
    ]{filtered_votes}
     {Id}
     {PostId->posts, UserId->users}
     {}
     {CreationDate}.*, 0, 91, DAYS)
    FOR EACH posts.* WHERE posts.PostTypeId == 1
                        AND posts.OwnerUserId IS NOT NULL
                        AND posts.OwnerUserId != -1
    ;
"""

In [44]:
check_correctness(dataset_stack, task_stack_post_votes, "train", query_orig, query_inj, type="tmp")

SQL query executed in 1.35 seconds.
SQL query executed in 1.27 seconds.
------------------- START TRAIN -------------------
FKeys without INJECTION: {'fk': 'posts'}
PKey without INJECTION: None
TimeCol without INJECTION: timestamp
FKeys with INJECTION: {'fk': 'posts'}
PKey with INJECTION: None
TimeCol with INJECTION: timestamp
Query without INJECTION:
 Empty DataFrame
Columns: [fk, timestamp, label, _merge]
Index: []
Query with INJECTION:
 Empty DataFrame
Columns: [fk, timestamp, label, _merge]
Index: []
In both:
              fk  timestamp  label _merge
0             0 2009-04-16      0   both
1             0 2009-07-16      0   both
2             0 2009-10-15      0   both
3             0 2010-01-14      0   both
4             0 2010-04-15      0   both
...         ...        ...    ...    ...
2453916  315139 2020-07-02      0   both
2453917  315140 2020-07-02      0   both
2453918  315142 2020-07-02      0   both
2453919  315144 2020-07-02      0   both
2453920  315145 2020-07-02   

In [45]:
query_orig = """
    PREDICT COUNT_DISTINCT(votes.* WHERE votes.votetypeid == 2)
    FOR EACH posts.* 
    ;
"""

In [46]:
query_inj = """
    PREDICT COUNT_DISTINCT(
    [
     SELECT 
        v.*
     FROM 
        votes v
     WHERE 
        v.votetypeid == 2
    ]{filtered_votes}
     {Id}
     {PostId->posts, UserId->users}
     {}.*)
    FOR EACH posts.*
    ;
"""

In [47]:
check_correctness(dataset_stack, None, "train", query_orig, query_inj, type="stat")

SQL query executed in 0.14 seconds.
SQL query executed in 0.12 seconds.
------------------- START TRAIN -------------------
FKeys without INJECTION: {'fk': 'posts'}
PKey without INJECTION: None
TimeCol without INJECTION: None
FKeys with INJECTION: {'fk': 'posts'}
PKey with INJECTION: None
TimeCol with INJECTION: None
Query without INJECTION:
 Empty DataFrame
Columns: [fk, label, _merge]
Index: []
Query with INJECTION:
 Empty DataFrame
Columns: [fk, label, _merge]
Index: []
In both:
             fk  label _merge
0            0     80   both
1            1     68   both
2            2     19   both
3            3      7   both
4            4     19   both
...        ...    ...    ...
262431  333886      3   both
262432  333887      1   both
262433  333888      1   both
262434  333891      1   both
262435  333892      1   both

[262436 rows x 3 columns]
------------------- END TRAIN ---------------------


In [48]:
query_orig = """
    PREDICT COUNT_DISTINCT(votes.* WHERE votes.votetypeid == 2, 0, 91, DAYS)
    FOR EACH POSTS.* WHERE posts.PostTypeId == 1
                       AND posts.OwnerUserId IS NOT NULL
                       AND posts.OwnerUserId != -1
    ;
"""

In [49]:
query_inj = """
    PREDICT COUNT_DISTINCT(votes.* WHERE votes.votetypeid == 2, 0, 91, DAYS)
    FOR EACH [
      SELECT
        p.*
      FROM
        posts p
      WHERE
        p.PostTypeId = ANY(ARRAY[1])
      AND
        p.OwnerUserId IS NOT NULL
      AND
        p.OwnerUserId != -1
    ]{filtered_posts}
     {Id}
     {Id->POSTS}
     {votes:PostId}
     {CreationDate}.*
    ;
"""

In [50]:
check_correctness(dataset_stack, task_stack_post_votes, "train", query_orig, query_inj, type="tmp")

SQL query executed in 1.36 seconds.
SQL query executed in 0.81 seconds.
------------------- START TRAIN -------------------
FKeys without INJECTION: {'fk': 'posts'}
PKey without INJECTION: None
TimeCol without INJECTION: timestamp
FKeys with INJECTION: {'fk': 'posts'}
PKey with INJECTION: None
TimeCol with INJECTION: timestamp
Query without INJECTION:
 Empty DataFrame
Columns: [fk, timestamp, label, _merge]
Index: []
Query with INJECTION:
 Empty DataFrame
Columns: [fk, timestamp, label, _merge]
Index: []
In both:
              fk  timestamp  label _merge
0             0 2009-04-16      0   both
1             0 2009-07-16      0   both
2             0 2009-10-15      0   both
3             0 2010-01-14      0   both
4             0 2010-04-15      0   both
...         ...        ...    ...    ...
2453916  315139 2020-07-02      0   both
2453917  315140 2020-07-02      0   both
2453918  315142 2020-07-02      0   both
2453919  315144 2020-07-02      0   both
2453920  315145 2020-07-02   

In [51]:
query_orig = """
    PREDICT COUNT_DISTINCT(votes.* WHERE votes.votetypeid == 2)
    FOR EACH POSTS.* WHERE posts.PostTypeId == 1
                       AND posts.OwnerUserId IS NOT NULL
                       AND posts.OwnerUserId != -1
    ;
"""

In [52]:
query_inj = """
    PREDICT COUNT_DISTINCT(votes.* WHERE votes.votetypeid == 2)
    FOR EACH [
      SELECT
        p.*
      FROM
        posts p
      WHERE
        p.PostTypeId = ANY(ARRAY[1])
      AND
        p.OwnerUserId IS NOT NULL
      AND
        p.OwnerUserId != -1
    ]{filtered_posts}
     {Id}
     {Id->POSTS}
     {votes:PostId}.*
    ;
"""

In [53]:
check_correctness(dataset_stack, None, "train", query_orig, query_inj, type="stat")

SQL query executed in 0.18 seconds.
SQL query executed in 0.26 seconds.
------------------- START TRAIN -------------------
FKeys without INJECTION: {'fk': 'posts'}
PKey without INJECTION: None
TimeCol without INJECTION: None
FKeys with INJECTION: {'fk': 'posts'}
PKey with INJECTION: None
TimeCol with INJECTION: None
Query without INJECTION:
 Empty DataFrame
Columns: [fk, label, _merge]
Index: []
Query with INJECTION:
 Empty DataFrame
Columns: [fk, label, _merge]
Index: []
In both:
             fk  label _merge
0            0     80   both
1           19     11   both
2           23     48   both
3           24     36   both
4           25     72   both
...        ...    ...    ...
123325  333880      2   both
123326  333883      1   both
123327  333886      3   both
123328  333887      1   both
123329  333891      1   both

[123330 rows x 3 columns]
------------------- END TRAIN ---------------------


In [54]:
query_orig = """
    PREDICT COUNT_DISTINCT(votes.* WHERE votes.votetypeid == 2, 0, 91, DAYS)
    FOR EACH posts.* WHERE posts.PostTypeId == 1
                        AND posts.OwnerUserId IS NOT NULL
                        AND posts.OwnerUserId != -1
    ;
"""

In [55]:
query_inj = """
    PREDICT COUNT_DISTINCT(
    [
     SELECT 
        v.*
     FROM 
        votes v
     WHERE 
        v.votetypeid == 2
    ]{filtered_votes}
     {Id}
     {PostId->filtered_posts, UserId->users}
     {}
     {CreationDate}.*, 0, 91, DAYS)
    FOR EACH [
      SELECT
        p.*
      FROM
        posts p
      WHERE
        p.PostTypeId == 1
      AND
        p.OwnerUserId IS NOT NULL
      AND
        p.OwnerUserId != ANY(ARRAY[-1])
    ]{filtered_posts}
     {Id}
     {Id->posts}
     {filtered_votes:PostId}
     {CreationDate}.*
    ;
"""

In [56]:
check_correctness(dataset_stack, task_stack_post_votes, "train", query_orig, query_inj, type="tmp")

SQL query executed in 1.30 seconds.
SQL query executed in 0.78 seconds.
------------------- START TRAIN -------------------
FKeys without INJECTION: {'fk': 'posts'}
PKey without INJECTION: None
TimeCol without INJECTION: timestamp
FKeys with INJECTION: {'fk': 'posts'}
PKey with INJECTION: None
TimeCol with INJECTION: timestamp
Query without INJECTION:
 Empty DataFrame
Columns: [fk, timestamp, label, _merge]
Index: []
Query with INJECTION:
 Empty DataFrame
Columns: [fk, timestamp, label, _merge]
Index: []
In both:
              fk  timestamp  label _merge
0             0 2009-04-16      0   both
1             0 2009-07-16      0   both
2             0 2009-10-15      0   both
3             0 2010-01-14      0   both
4             0 2010-04-15      0   both
...         ...        ...    ...    ...
2453916  315139 2020-07-02      0   both
2453917  315140 2020-07-02      0   both
2453918  315142 2020-07-02      0   both
2453919  315144 2020-07-02      0   both
2453920  315145 2020-07-02   

In [57]:
query_orig = """
    PREDICT COUNT_DISTINCT(votes.* WHERE votes.votetypeid == 2)
    FOR EACH posts.* WHERE posts.PostTypeId == 1
                        AND posts.OwnerUserId IS NOT NULL
                        AND posts.OwnerUserId != -1
    ;
"""

In [58]:
query_inj = """
    PREDICT COUNT_DISTINCT(
    [
     SELECT 
        v.*
     FROM 
        votes v
     WHERE 
        v.votetypeid == 2
    ]{filtered_votes}
     {Id}
     {PostId->filtered_posts, UserId->users}
     {}.*)
    FOR EACH [
      SELECT
        p.*
      FROM
        posts p
      WHERE
        p.PostTypeId == 1
      AND
        p.OwnerUserId IS NOT NULL
      AND
        p.OwnerUserId != ANY(ARRAY[-1])
    ]{filtered_posts}
     {Id}
     {Id->posts}
     {filtered_votes:PostId}.*
    ;
"""

In [59]:
check_correctness(dataset_stack, None, "train", query_orig, query_inj, type="stat")

SQL query executed in 0.16 seconds.
SQL query executed in 0.26 seconds.
------------------- START TRAIN -------------------
FKeys without INJECTION: {'fk': 'posts'}
PKey without INJECTION: None
TimeCol without INJECTION: None
FKeys with INJECTION: {'fk': 'posts'}
PKey with INJECTION: None
TimeCol with INJECTION: None
Query without INJECTION:
 Empty DataFrame
Columns: [fk, label, _merge]
Index: []
Query with INJECTION:
 Empty DataFrame
Columns: [fk, label, _merge]
Index: []
In both:
             fk  label _merge
0            0     80   both
1           19     11   both
2           23     48   both
3           24     36   both
4           25     72   both
...        ...    ...    ...
123325  333880      2   both
123326  333883      1   both
123327  333886      3   both
123328  333887      1   both
123329  333891      1   both

[123330 rows x 3 columns]
------------------- END TRAIN ---------------------


## F1 Dataset

In [60]:
dataset_f1 = get_dataset(name="rel-f1", download=False)
db_f1 = dataset_f1.get_db()

Loading Database object from /home/kolesiko/.cache/relbench/rel-f1/db...
Done in 0.07 seconds.


In [27]:
db_f1.table_dict['races']

Table(df=
     raceId  year  round  circuitId                  name                date  \
0         0  1950      1          8    British Grand Prix 1950-05-13 00:00:00   
1         1  1950      2          5     Monaco Grand Prix 1950-05-21 00:00:00   
2         2  1950      3         18      Indianapolis 500 1950-05-30 00:00:00   
3         3  1950      4         65      Swiss Grand Prix 1950-06-04 00:00:00   
4         4  1950      5         12    Belgian Grand Prix 1950-06-18 00:00:00   
..      ...   ...    ...        ...                   ...                 ...   
815     815  2009     13         13    Italian Grand Prix 2009-09-13 12:00:00   
816     816  2009     14         14  Singapore Grand Prix 2009-09-27 12:00:00   
817     817  2009     15         21   Japanese Grand Prix 2009-10-04 05:00:00   
818     818  2009     16         17  Brazilian Grand Prix 2009-10-18 16:00:00   
819     819  2009     17         23  Abu Dhabi Grand Prix 2009-11-01 11:00:00   

         time  
0

In [28]:
db_f1.table_dict['constructors']

Table(df=
     constructorId constructorRef            name nationality
0                0        mclaren         McLaren     British
1                1     bmw_sauber      BMW Sauber      German
2                2       williams        Williams     British
3                3        renault         Renault      French
4                4     toro_rosso      Toro Rosso     Italian
..             ...            ...             ...         ...
206            206          manor  Manor Marussia     British
207            207           haas    Haas F1 Team    American
208            208   racing_point    Racing Point     British
209            209     alphatauri      AlphaTauri     Italian
210            210         alpine  Alpine F1 Team      French

[211 rows x 4 columns],
  fkey_col_to_pkey_table={},
  pkey_col=constructorId,
  time_col=None)

In [29]:
db_f1.table_dict['constructor_results']

Table(df=
      constructorResultsId  raceId  constructorId  points                date
0                        0      48            103    13.0 1956-01-22 00:00:00
1                        1      48              5    12.0 1956-01-22 00:00:00
2                        2      54            126     0.0 1956-08-05 00:00:00
3                        3      54            103    15.0 1956-08-05 00:00:00
4                        4      54              5     9.0 1956-08-05 00:00:00
...                    ...     ...            ...     ...                 ...
9403                  9403     819              5     0.0 2009-11-01 11:00:00
9404                  9404     819              0     0.0 2009-11-01 11:00:00
9405                  9405     819              2     0.0 2009-11-01 11:00:00
9406                  9406     819              4     1.0 2009-11-01 11:00:00
9407                  9407     819              6     5.0 2009-11-01 11:00:00

[9408 rows x 5 columns],
  fkey_col_to_pkey_table={'r

In [61]:
sconverter_f1 = SConverter(db_f1)
tconverter_f1 = TConverter(db_f1, get_timestamps(dataset_f1, pd.Timedelta(days=30), 10, "train"))

Loading Database object from /home/kolesiko/.cache/relbench/rel-f1/db...
Done in 0.03 seconds.


In [62]:
query_orig = """
    PREDICT SUM(races.round, 0, 30, DAYS)
    FOR EACH constructors.* 
    WHERE constructors.constructorId > 20
              AND constructors.constructorId < 40
    ;
"""

In [63]:
query_inj = """
    PREDICT SUM(
        [SELECT 
           r.round,
           cs.constructorStandingsId,
           cs.constructorId,
           r.date
         FROM
           races r
         JOIN 
           constructor_standings cs
         ON 
           r.raceId = cs.raceId
        ]{constructor_races}
         {constructorStandingsId}
         {constructorId->constructors}
         {}
         {date}.round, 0, 30, DAYS)
    FOR EACH constructors.* 
    WHERE constructors.constructorId > 20
              AND constructors.constructorId < 40
    ;
"""

In [64]:
merge_dataframes(
    tconverter_f1.convert(query_orig, execute=True).df,
    tconverter_f1.convert(query_inj, execute=True).df,
    type="tmp"
)

SQL query executed in 0.07 seconds.
SQL query executed in 0.17 seconds.
Query without INJECTION:
 Empty DataFrame
Columns: [fk, timestamp, label, _merge]
Index: []
Query with INJECTION:
 Empty DataFrame
Columns: [fk, timestamp, label, _merge]
Index: []
In both:
       fk  timestamp  label _merge
0     21 1986-03-12    1.0   both
1     21 1986-04-11    9.0   both
2     21 1986-05-11    5.0   both
3     21 1986-06-10   21.0   both
4     21 1986-07-10   19.0   both
...   ..        ...    ...    ...
1451  39 1991-06-14   21.0   both
1452  39 1991-07-14   19.0   both
1453  39 1991-08-13   23.0   both
1454  39 1991-09-12   27.0   both
1455  39 1991-10-12   31.0   both

[1456 rows x 4 columns]


In [65]:
query_orig = """
    PREDICT SUM(races.round)
    FOR EACH constructors.* 
    WHERE constructors.constructorId > 20
              AND constructors.constructorId < 40
    ;
"""

In [66]:
query_inj = """
    PREDICT SUM(
        [SELECT 
           r.round,
           cs.constructorStandingsId,
           cs.constructorId,
           r.date
         FROM
           races r
         JOIN 
           constructor_standings cs
         ON 
           r.raceId = cs.raceId
        ]{constructor_races}
         {constructorStandingsId}
         {constructorId->constructors}
         {}.round)
    FOR EACH constructors.* 
    WHERE constructors.constructorId > 20
              AND constructors.constructorId < 40
    ;
"""

In [67]:
merge_dataframes(
    sconverter_f1.convert(query_orig, execute=True).df,
    sconverter_f1.convert(query_inj, execute=True).df,
    type="stat"
)

SQL query executed in 0.02 seconds.
SQL query executed in 0.02 seconds.
Query without INJECTION:
 Empty DataFrame
Columns: [fk, label, _merge]
Index: []
Query with INJECTION:
 Empty DataFrame
Columns: [fk, label, _merge]
Index: []
In both:
     fk   label _merge
0   21  2243.0   both
1   22   153.0   both
2   23   421.0   both
3   24  3604.0   both
4   25  1032.0   both
5   26  2810.0   both
6   27   288.0   both
7   28   830.0   both
8   29   289.0   both
9   30   288.0   both
10  31  3258.0   both
11  32  1086.0   both
12  33  2523.0   both
13  34   679.0   both
14  35   236.0   both
15  36  1597.0   both
16  38   738.0   both
17  39   136.0   both
